In [ ]:
from utils import *
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
import json

In [ ]:
def adaptive_ratio_v1_0_loader():
    results_dir = Path("../results/adaptive_ratio_v1_0")
    scalars = TBScalars(".cache/adaptive_ratio_v1_0")

    res_df = []
    for test in tqdm([*results_dir.iterdir()]):
        params = test.name.split("-")
        test_r = {}
        test_r["env"] = params[0]
        types = {"seed": int}
        for (name, typ), value in zip(types.items(), params[1:]):
            test_r[name] = typ(value.removeprefix(f"{name}="))
        df = scalars.read(test)
        scores = df[df["tag"] == "val/mean_ep_ret"]["value"]
        test_r["score"] = scores.iloc[-1]
        res_df.append({"path": test, **test_r})
    res_df = pd.DataFrame.from_records(res_df)
    res_df

    return res_df, scalars


def adaptive_wm_ratio_v1_0_loader():
    results_dir = Path("../results/adaptive_wm_ratio_v1_0")
    scalars = TBScalars(".cache/adaptive_wm_ratio_v1_0")

    res_df = []
    for test in tqdm([*results_dir.iterdir()]):
        params = test.name.split("-")
        test_r = {}
        test_r["env"] = params[0]
        types = {"rl_ratio": int, "seed": int}
        for (name, typ), value in zip(types.items(), params[1:]):
            test_r[name] = typ(value.removeprefix(f"{name}="))
        df = scalars.read(test)
        scores = df[df["tag"] == "val/mean_ep_ret"]["value"]
        test_r["score"] = scores.iloc[-1]
        res_df.append({"path": test, **test_r})
    res_df = pd.DataFrame.from_records(res_df)
    res_df

    return res_df, scalars


def adaptive_wm_ratio_v1_1_loader():
    res_df = []

    results_dir = Path("../results/adaptive_wm_ratio_v1_1")
    scalars = TBScalars(".cache/adaptive_wm_ratio_v1_1")
    for test in tqdm([*results_dir.iterdir()]):
        params = test.name.split("-")
        test_r = {}
        test_r["env"] = params[0]
        types = {"rl_ratio": int, "seed": int}
        for (name, typ), value in zip(types.items(), params[1:]):
            test_r[name] = typ(value.removeprefix(f"{name}="))
        df = scalars.read(test)
        scores = df[df["tag"] == "val/mean_ep_ret"]["value"]
        test_r["score"] = scores.iloc[-1]
        res_df.append({"path": test, **test_r})

    res_df = pd.DataFrame.from_records(res_df)

    return res_df, scalars


res_v10_df, scalars_v10 = adaptive_ratio_v1_0_loader()
res_wm_v10_df, scalars_wm_v10 = adaptive_wm_ratio_v1_0_loader()
res_wm_v11_df, scalars_wm_v11 = adaptive_wm_ratio_v1_1_loader()
res_dfs = {"v1.0": res_v10_df, "v1.0-wm": res_wm_v10_df, "v1.1-wm": res_wm_v11_df}

In [ ]:
df = scalars_wm_v11.read(res_wm_v11_df.iloc[29]["path"])
val_loss = df[df["tag"] == "ada/val_loss"]
px.line(val_loss, x="step", y="value", log_y=True)

In [ ]:
from scipy.stats import norm


def mann_kendall(values: np.ndarray):
    S = np.tril(np.subtract.outer(values, values)).sum()
    n = len(values)
    V = 1 / 18.0 * (n * (n - 1) * (2 * n + 5))  # Assuming no ties
    Z = (S - np.sign(S)) / np.sqrt(V)
    return norm.cdf(Z)


v = val_loss[(val_loss["step"] >= 50e3) & (val_loss["step"] < 275e3)]
print(mann_kendall(v["value"].to_numpy()))
px.line(v, x="step", y="value")